In [65]:
import pandas as pd
import numpy as np
import torch
import os
import sys
import numpy as np

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)
 
from syn_project.utils_train import *
from syn_project.utils_color_analysis import *
from syn_project.utils_notebook import *

# conditions = [
#     "ablation_cont_cy",
#     "ablation_cont_cy_dcy",
#     "ablation_cont_cy_dcy_trans",
#     "ablation_cont_cy_trans",
#     "ablation_cont_dcy",
#     "ablation_cont_dcy_trans",
#     "ablation_cont_trans",
#     "ablation_cy",
#     "ablation_cy_dcy",
#     "ablation_cy_dcy_trans",
#     "ablation_cy_trans",
#     "ablation_dcy",
#     "ablation_dcy_trans",
#     "ablation_trans",
# ]


conditions = [
      "ablation_action_cy_dcy_cont_trans",
      "ablation_action_cy_cont_trans",
      "ablation_action_cy_dcy_trans",
      "ablation_action_cy_dcy_cont",
      "ablation_action_cy_trans",
      "ablation_action_cy_cont",
      "ablation_action_cy_dcy",
      "ablation_action_cy",
      "ablation_action_dcy_cont_trans",
      "ablation_action_dcy_cont",
     "ablation_action_dcy_trans",
    "ablation_action_dcy",
     "ablation_action_cont_trans",
     "ablation_action_trans",
     "ablation_action_cont",
     #"ablation_action_cy_hidcy_hitrans2",

    "ablation_cont_cy",
    "ablation_cont_cy_dcy",
    "ablation_cont_cy_dcy_trans",
    "ablation_cont_cy_trans",
    "ablation_cont_dcy",
    "ablation_cont_dcy_trans",
    "ablation_cont_trans",
    "ablation_cy",
    "ablation_cy_hicont",
    "ablation_cy_dcy",
    "ablation_cy_dcy_trans",
    #"ablation_cy_hidcy_hitrans",
    #"ablation_cy_hidcy_trans",
    #"ablation_cy_hidcy_hitrans2",
    "ablation_cy_trans",
    "ablation_dcy",
    "ablation_dcy_trans",
    "ablation_trans",
]

start_vision = [True]

In [66]:


checkpoint_epoch  = 0
n_samples_test    = 1000
split             = "test"
dataset           = "biased_00"

rows = []

for condition in conditions:
    print(f"\n=== {condition} ===")
    with total_silence():
        (global_workspace, domain_mods, gw_mod,
         visual_module, original_data,
         latent_domains, modules_name) = get_modules_data_from_exp(
            experiment_name=condition,
            n_samples_test=n_samples_test,
            split=split,
            checkpoint_epoch=checkpoint_epoch,
        )

    for s in start_vision:
        objects = get_objects_from_v_latents(
            latent_domains, gw_mod, global_workspace, modules_name,
            modality_from='attr', modality_through='color',
            modality_main=['attr'], modality_add='color',
            start_v=s,
        )

        original_images_rgb = visual_module.decode_images(original_data['v_latents'])

        cat = []
        if 'attr' in modules_name:
            cat = original_data['attr'][0]
        if 'cat' in modules_name:
            cat = original_data['cat']

        # vision2 = la reconstruction finale qui t'intéresse
        decoded_images = visual_module.decode_images(objects['vision2'])

        # --- qualité de reconstruction ---
        recon = compute_reconstruction_quality(original_images_rgb, decoded_images)

        # --- analyse couleur / LDA ---
        colors_np = np.clip((objects['x2']['color'].detach().cpu().numpy() + 1) / 2, 0, 1) 
        cats      = cat.argmax(dim=1).detach().cpu().numpy()
        metrics, _ = hue_analysis(colors_np, cats,
                               value=0.75, saturation_boost=1.8)
        results = logistic_probe(colors_np, cats)

        rows.append({
            "condition"         : condition,
            "start_v"           : s,
            "ssim"              : recon["ssim"],
            "lda_score"         : metrics["lda_score"],
        })

        del decoded_images, objects   # libère la mémoire GPU
        torch.cuda.empty_cache()

# ---- tableau récapitulatif ----

df = pd.DataFrame(rows).set_index("condition")
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)


=== ablation_action_cy_dcy_cont_trans ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_action_cy_cont_trans ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_action_cy_dcy_trans ===


...


=== ablation_dcy_trans ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)



=== ablation_trans ===


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/torch/nn/modules/module.py:1736: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


,start_v,ssim,lda_score
condition,,,
ablation_action_cy_dcy_cont_trans,True,0.884,0.961
ablation_action_cy_cont_trans,True,0.858,0.943
ablation_action_cy_dcy_trans,True,0.841,0.742
ablation_action_cy_dcy_cont,True,0.825,0.813
ablation_action_cy_trans,True,0.669,0.926
ablation_action_cy_cont,True,0.785,0.440
ablation_action_cy_dcy,True,0.851,0.349
ablation_action_cy,True,0.783,0.370
ablation_action_dcy_cont_trans,True,0.850,0.397


In [67]:
rows

[{'condition': 'ablation_action_cy_dcy_cont_trans',
  'start_v': True,
  'ssim': 0.8840504288673401,
  'lda_score': 0.961},
 {'condition': 'ablation_action_cy_cont_trans',
  'start_v': True,
  'ssim': 0.8577839732170105,
  'lda_score': 0.943},
 {'condition': 'ablation_action_cy_dcy_trans',
  'start_v': True,
  'ssim': 0.8407487869262695,
  'lda_score': 0.742},
 {'condition': 'ablation_action_cy_dcy_cont',
  'start_v': True,
  'ssim': 0.8251867294311523,
  'lda_score': 0.813},
 {'condition': 'ablation_action_cy_trans',
  'start_v': True,
  'ssim': 0.6693763136863708,
  'lda_score': 0.926},
 {'condition': 'ablation_action_cy_cont',
  'start_v': True,
  'ssim': 0.7847457528114319,
  'lda_score': 0.44},
 {'condition': 'ablation_action_cy_dcy',
  'start_v': True,
  'ssim': 0.8506579995155334,
  'lda_score': 0.349},
 {'condition': 'ablation_action_cy',
  'start_v': True,
  'ssim': 0.7834728956222534,
  'lda_score': 0.37},
 {'condition': 'ablation_action_dcy_cont_trans',
  'start_v': True,
  

In [68]:
df = pd.DataFrame(rows)
# df = df.sort_values("lda_score", ascending=False)   # tri par LDA, ajuste si besoin

df = df[(df.ssim > 0.8) & (df.start_v == True) ]
# affichage Jupyter
display(
    df.style
      .format({"ssim": "{:.3f}",
               "lda_score": "{:.3f}"})
      .background_gradient(subset=["ssim", "lda_score",], cmap="RdYlGn")
)

,condition,start_v,ssim,lda_score
0,ablation_action_cy_dcy_cont_trans,True,0.884,0.961
1,ablation_action_cy_cont_trans,True,0.858,0.943
2,ablation_action_cy_dcy_trans,True,0.841,0.742
3,ablation_action_cy_dcy_cont,True,0.825,0.813
6,ablation_action_cy_dcy,True,0.851,0.349
8,ablation_action_dcy_cont_trans,True,0.850,0.397
9,ablation_action_dcy_cont,True,0.812,0.491
10,ablation_action_dcy_trans,True,0.811,0.430
16,ablation_cont_cy_dcy,True,0.861,0.758
17,ablation_cont_cy_dcy_trans,True,0.891,0.975


In [69]:
df

,condition,start_v,ssim,lda_score
0,ablation_action_cy_dcy_cont_trans,True,0.884050,0.961
1,ablation_action_cy_cont_trans,True,0.857784,0.943
2,ablation_action_cy_dcy_trans,True,0.840749,0.742
3,ablation_action_cy_dcy_cont,True,0.825187,0.813
6,ablation_action_cy_dcy,True,0.850658,0.349
8,ablation_action_dcy_cont_trans,True,0.850031,0.397
9,ablation_action_dcy_cont,True,0.812461,0.491
10,ablation_action_dcy_trans,True,0.810898,0.430
16,ablation_cont_cy_dcy,True,0.860693,0.758
17,ablation_cont_cy_dcy_trans,True,0.890742,0.975


In [70]:
rows

[{'condition': 'ablation_action_cy_dcy_cont_trans',
  'start_v': True,
  'ssim': 0.8840504288673401,
  'lda_score': 0.961},
 {'condition': 'ablation_action_cy_cont_trans',
  'start_v': True,
  'ssim': 0.8577839732170105,
  'lda_score': 0.943},
 {'condition': 'ablation_action_cy_dcy_trans',
  'start_v': True,
  'ssim': 0.8407487869262695,
  'lda_score': 0.742},
 {'condition': 'ablation_action_cy_dcy_cont',
  'start_v': True,
  'ssim': 0.8251867294311523,
  'lda_score': 0.813},
 {'condition': 'ablation_action_cy_trans',
  'start_v': True,
  'ssim': 0.6693763136863708,
  'lda_score': 0.926},
 {'condition': 'ablation_action_cy_cont',
  'start_v': True,
  'ssim': 0.7847457528114319,
  'lda_score': 0.44},
 {'condition': 'ablation_action_cy_dcy',
  'start_v': True,
  'ssim': 0.8506579995155334,
  'lda_score': 0.349},
 {'condition': 'ablation_action_cy',
  'start_v': True,
  'ssim': 0.7834728956222534,
  'lda_score': 0.37},
 {'condition': 'ablation_action_dcy_cont_trans',
  'start_v': True,
  

In [71]:
def filter_results(data, action=None, min_ssim=None, sort_desc=True):
    results = data

    # Filtre sur la présence de "action"
    if action is not None:
        results = [
            d for d in results
            if ("action" in d["condition"]) == action
        ]

    # Filtre sur le SSIM minimum
    if min_ssim is not None:
        results = [
            d for d in results
            if d["ssim"] >= min_ssim
        ]

    # Tri sur le score LDA
    results = sorted(
        results,
        key=lambda d: d["lda_score"],
        reverse=sort_desc
    )

    return results

In [76]:
res = filter_results(
    rows,
    action=True,
    min_ssim=0.7,
)

In [77]:
def condition_to_flags(condition, loss_names=("cy", "dcy", "cont", "trans")):
    tokens = set(condition.split("_"))
    return {name: int(name in tokens) for name in loss_names}

def to_latex_table(data, loss_names=("cy", "dcy", "cont", "trans")):
    header = " & ".join(loss_names) + " & LDA & SSIM \\\\"
    rows = []
    for d in data:
        flags = condition_to_flags(d["condition"], loss_names)
        flag_str = " & ".join(str(flags[n]) for n in loss_names)
        rows.append(f"{flag_str} & {d['lda_score']:.3f} & {d['ssim']:.3f} \\\\")

    body = "\n".join(rows)
    ncols = len(loss_names) + 2
    col_spec = "c" * len(loss_names) + " cc"

    return f"""\\begin{{table}}[htbp]
\\centering
\\begin{{tabular}}{{{col_spec}}}
\\toprule
{header}
\\midrule
{body}
\\bottomrule
\\end{{tabular}}
\\caption{{Résultats d'ablation des pertes.}}
\\label{{tab:ablation_losses}}
\\end{{table}}"""

print(to_latex_table(res))

\begin{table}[htbp]
\centering
\begin{tabular}{cccc cc}
\toprule
cy & dcy & cont & trans & LDA & SSIM \\
\midrule
1 & 1 & 1 & 1 & 0.961 & 0.884 \\
1 & 0 & 1 & 1 & 0.943 & 0.858 \\
1 & 1 & 1 & 0 & 0.813 & 0.825 \\
1 & 1 & 0 & 1 & 0.742 & 0.841 \\
0 & 1 & 1 & 0 & 0.491 & 0.812 \\
1 & 0 & 1 & 0 & 0.440 & 0.785 \\
0 & 1 & 0 & 1 & 0.430 & 0.811 \\
0 & 0 & 1 & 1 & 0.413 & 0.715 \\
0 & 1 & 1 & 1 & 0.397 & 0.850 \\
1 & 0 & 0 & 0 & 0.370 & 0.783 \\
1 & 1 & 0 & 0 & 0.349 & 0.851 \\
\bottomrule
\end{tabular}
\caption{Résultats d'ablation des pertes.}
\label{tab:ablation_losses}
\end{table}
